In [7]:
import onnx
from onnx import helper, TensorProto
# Inputs
a = helper.make_tensor_value_info("A", TensorProto.FLOAT, [10, 10])
b = helper.make_tensor_value_info("B", TensorProto.FLOAT, [10, 10])
d = helper.make_tensor_value_info("D", TensorProto.FLOAT, [10, 10])
# Output
y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, [10, 10])
# Nodes
matmul_node = helper.make_node("MatMul", inputs=["A", "B"], outputs=["Y"])
#add_node = helper.make_node("Add", inputs=["AB", "D"], outputs=["Y"])
# Graph & Model
graph = helper.make_graph(
    [matmul_node],
    "gemm_add",
    inputs=[a, b],
    outputs=[y],
)

# going to export model here
model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 21)])
model.ir_version = 10
onnx.checker.check_model(model)
onnx.save(model, "gemm_add.onnx")

print(onnx.printer.to_text(model))

<
   ir_version: 10,
   opset_import: ["" : 21]
>
gemm_add (float[10,10] A, float[10,10] B) => (float[10,10] Y) {
   Y = MatMul (A, B)
}


# Using ONNX-MLIR

Starting from the `.onnx` file from above, we can run the following `/home/vhe/frameworks/onnx-mlir/build/Release/bin/onnx-mlir gemm_add.onnx --EmitONNXBasic` to get the following IR:

```
module attributes {llvm.data_layout = "e-m:e-p270:32:32-p271:32:32-p272:64:64-i64:64-i128:128-f80:128-n8:16:32:64-S128", llvm.target_triple = "x86_64-unknown-linux-gnu", "onnx-mlir.symbol-postfix" = "gemm_add"} {
  func.func @main_graph(%arg0: tensor<10x10xf32> {onnx.name = "A"}, %arg1: tensor<10x10xf32> {onnx.name = "B"}) -> (tensor<10x10xf32> {onnx.name = "Y"}) {
    %0 = "onnx.MatMul"(%arg0, %arg1) {onnx_node_name = "onnx.MatMul_0"} : (tensor<10x10xf32>, tensor<10x10xf32>) -> tensor<10x10xf32>
    return %0 : tensor<10x10xf32>
  }
  "onnx.EntryPoint"() <{func = @main_graph}> : () -> ()
}
```

Then we can run the following to convert the ONNX IR into Linalg ` /home/vhe/frameworks/onnx-mlir/build/Release/bin/onnx-mlir-opt gemm_add.onnx.mlir --convert-onnx-to-linalg`: 

```
module attributes {llvm.data_layout = "e-m:e-p270:32:32-p271:32:32-p272:64:64-i64:64-i128:128-f80:128-n8:16:32:64-S128", llvm.target_triple = "x86_64-unknown-linux-gnu", "onnx-mlir.symbol-postfix" = "gemm_add"} {
  func.func @main_graph(%arg0: tensor<10x10xf32> {onnx.name = "A"}, %arg1: tensor<10x10xf32> {onnx.name = "B"}) -> (tensor<10x10xf32> {onnx.name = "Y"}) {
    %cst = arith.constant 0.000000e+00 : f32
    %0 = tensor.empty() : tensor<10x10xf32>
    %1 = linalg.fill ins(%cst : f32) outs(%0 : tensor<10x10xf32>) -> tensor<10x10xf32>
    %2 = linalg.matmul ins(%arg0, %arg1 : tensor<10x10xf32>, tensor<10x10xf32>) outs(%1 : tensor<10x10xf32>) -> tensor<10x10xf32>
    return %2 : tensor<10x10xf32>
  }
  "onnx.EntryPoint"() <{func = @main_graph}> : () -> ()
}
```

Deleting all the `onnx` dependencies manually, we have: 

```
module attributes {llvm.data_layout = "e-m:e-p270:32:32-p271:32:32-p272:64:64-i64:64-i128:128-f80:128-n8:16:32:64-S128", llvm.target_triple = "x86_64-unknown-linux-gnu", "onnx-mlir.symbol-postfix" = "gemm_add"} {
  func.func @main_graph(%arg0: tensor<10x10xf32>, %arg1: tensor<10x10xf32>) -> (tensor<10x10xf32>) {
    %cst = arith.constant 0.000000e+00 : f32
    %0 = tensor.empty() : tensor<10x10xf32>
    %1 = linalg.fill ins(%cst : f32) outs(%0 : tensor<10x10xf32>) -> tensor<10x10xf32>
    %2 = linalg.matmul ins(%arg0, %arg1 : tensor<10x10xf32>, tensor<10x10xf32>) outs(%1 : tensor<10x10xf32>) -> tensor<10x10xf32>
    return %2 : tensor<10x10xf32>
  }
}
```

We then use rocmlir-driver and rocmlir-gen to compile the mlir from above using:

```
~/rocMLIR/build-release/bin/rocmlir-gen rocmlir.mlir -arch=gfx950 -fut main_graph -clone-harness | ~/rocMLIR/build-release/bin/rocmlir-driver --kernel-pipeline=highlevel
```

Output: 

```
#map = affine_map<(d0, d1) -> (d0, d1)>
#map1 = affine_map<(d0, d1, d2) -> (d0, d2)>
#map2 = affine_map<(d0, d1, d2) -> (d2, d1)>
#map3 = affine_map<(d0, d1, d2) -> (d0, d1)>
module attributes {llvm.data_layout = "e-m:e-p270:32:32-p271:32:32-p272:64:64-i64:64-i128:128-f80:128-n8:16:32:64-S128", llvm.target_triple = "x86_64-unknown-linux-gnu", "onnx-mlir.symbol-postfix" = "gemm_add"} {
  func.func @main_graph(%arg0: memref<10x10xf32> {mhal.read_access}, %arg1: memref<10x10xf32> {mhal.read_access}, %arg2: memref<10x10xf32> {mhal.write_access}) {
    %cst = arith.constant 0.000000e+00 : f32
    %alloc = memref.alloc() {alignment = 64 : i64} : memref<10x10xf32>
    linalg.generic {indexing_maps = [#map], iterator_types = ["parallel", "parallel"]} outs(%alloc : memref<10x10xf32>) {
    ^bb0(%out: f32):
      linalg.yield %cst : f32
    }
    linalg.generic {indexing_maps = [#map1, #map2, #map3], iterator_types = ["parallel", "parallel", "reduction"]} ins(%arg0, %arg1 : memref<10x10xf32>, memref<10x10xf32>) outs(%alloc : memref<10x10xf32>) {
    ^bb0(%in: f32, %in_0: f32, %out: f32):
      %0 = arith.mulf %in, %in_0 : f32
      %1 = arith.addf %out, %0 : f32
      linalg.yield %1 : f32
    }
    memref.copy %alloc, %arg2 : memref<10x10xf32> to memref<10x10xf32>
    return
  }
  func.func @main_graph_wrapper(%arg0: memref<10x10xf32>, %arg1: memref<10x10xf32>, %arg2: memref<10x10xf32>) {
    %alloc = memref.alloc() : memref<10x10xf32>
    %token = mhal.launch @main_graph (%arg0, %arg1, %alloc) : (memref<10x10xf32>, memref<10x10xf32>, memref<10x10xf32>)
    mhal.await %token : !mhal.token
    memref.copy %alloc, %arg2 : memref<10x10xf32> to memref<10x10xf32>
    return
  }
  module @__xmodule_ attributes {mhal.arch = "gfx950", mhal.module} {
    func.func @main_graph(%arg0: memref<10x10xf32> {mhal.read_access}, %arg1: memref<10x10xf32> {mhal.read_access}, %arg2: memref<10x10xf32> {mhal.write_access}) attributes {kernel, original_func = @main_graph} {
      %alloc = memref.alloc() {alignment = 64 : i64} : memref<10x10xf32>
      rock.gemm %alloc = %arg0 * %arg1 storeMethod =  set : memref<10x10xf32> = memref<10x10xf32> * memref<10x10xf32>
      memref.copy %alloc, %arg2 : memref<10x10xf32> to memref<10x10xf32>
      return
    }
  }
}
```
